In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import lbpscreen as lbp
from pathlib import Path

MODEL_DIR = Path(r"C:\Users\IPRDC\lbp-screen\data\processed\core_models")
FILE_MAP = {"E_faecalis_MMH594":    "Enterococcus_faecalis_MMH594_AGORA2_v2.01.xml",
            "E_lenta_DSM2243":      "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml",
            "E_coli_Nissle1917":    "Escherichia_coli_Nissle_1917_AGORA2_v2.01.xml",
            "A_muciniphila_BAA835": "Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml",
}

models = lbp.load_models(MODEL_DIR, FILE_MAP)
m = models["E_coli_Nissle1917"]

## objective_reaction

**Purpose**: FBA maximizes one reaction, the objective, which usually biomass. Every diagnostic function needs to know which reaction that is. Its name differs between models (e.g., bio1, biomass467, etc.), so we find it automatically instead of typing it.

**How the model marks it?**
Every reaction has an objective_coefficient:
coefficient = 0 : ordinary reactions
coefficient = 1 : reaction being maximized

In [2]:
r = m.reactions[0]
print(r.id)
print(r.objective_coefficient)
print(len(m.reactions))

2HBO
0.0
3574


In [3]:
for r in m.reactions[:5]:
    print(r.id, r.objective_coefficient)

2HBO 0.0
34DHOXPEGOX 0.0
34HPPOR 0.0
3HPPD 0.0
5HLTDL 0.0


In [4]:
for r in m.reactions:
    if r.objective_coefficient != 0:
        print(r.id, r.objective_coefficient)

bio1 1.0


In [5]:
found = []
for r in m.reactions:
    if r.objective_coefficient != 0:
        found.append(r)
print(found)

[<Reaction bio1 at 0x211538dfe50>]


In [6]:
found = [r for r in m.reactions if r.objective_coefficient != 0]
print(found)

[<Reaction bio1 at 0x211538dfe50>]


In [7]:
print(len(found))
print(found[0].id)

1
bio1


In [8]:
def count_reactions(model):
    return len(model.reactions)

print(count_reactions(m))
print(count_reactions(models["E_lenta_DSM2243"]))

3574
1035


In [9]:
for label, model in models.items():
    bio = lbp.objective_reaction(model)
    print(f"{label:22s}: {bio.id}")
    print(bio)

E_faecalis_MMH594     : bio1
bio1: 0.0079397 10fthf_c + 0.0079397 2dmmq8_c + 0.0079397 5mthf_c + 0.0079397 ACP_c + 0.0018061 PGP_c + 0.0079397 adocbl_c + 0.0018061 ai17tca1_c + 0.0018061 ai17tcaacgam_c + 0.0018061 ai17tcaala_D_c + 0.0018061 ai17tcaglc_c + 0.26756 ala_L_c + 0.0079397 amet_c + 0.1934 arg_L_c + 0.14833 asn_L_c + 0.14833 asp_L_c + 41.2914 atp_c + 0.0079397 ca2_c + 0.0079397 cl_c + 0.0085871 clpn180_c + 0.0085871 clpnai17_c + 0.0085871 clpni17_c + 0.0079397 coa_c + 0.0079397 cobalt2_c + 0.026124 ctp_c + 0.0079397 cu2_c + 0.056954 cys_L_c + 0.016932 datp_c + 0.0099445 dctp_c + 0.0099445 dgtp_c + dnarep_c + 0.016932 dttp_c + 0.0079397 fad_c + 0.0079397 fe2_c + 0.0079397 fe3_c + 0.261 gln_L_c + 0.261 glu_L_c + 0.40933 gly_c + 0.0018061 glyc45tca_c + 0.0018061 glyc45tcaala_D_c + 0.0018061 glyc45tcaglc_c + 0.0079397 gthrd_c + 0.042403 gtp_c + 41.257 h_c + 0.081948 his_L_c + 0.0018061 i17tca1_c + 0.0018061 i17tcaacgam_c + 0.0018061 i17tcaala_D_c + 0.0018061 i17tcaglc_c + 0.27043 

## boundary_census

**Purpose**: A boundary reaction moves a metabolite into or out of the model. COBRApy sorts boundary reactions into types, but model.exchanges lists only one type. This function lists every boundary reaction, its type and its bounds

**COBRApy Types of Boundary Reactions**
1. Exchange reactions - metabolite is outside th cell
2. Demand reactions - metabolite is inside the cell
3. Sink reactions - metabolite is inside the cell
4. Other reactions - metabolite does not fit COBRApy's rules

In [10]:
print(len(m.boundary))
print(len(m.exchanges))

589
569


In [11]:
r = m.exchanges[0]
print(r.metabolites)
met = list(r.metabolites)[0]
print(met.id, met.compartment)

{<Metabolite 5mthf_e at 0x2114e3fde90>: -1.0}
5mthf_e e


In [12]:
exchange_ids = {r.id for r in m.exchanges}
demand_ids = {r.id for r in m.demands}
sink_ids = {r.id for r in m.sinks}
print("Exchange reactions:", len(exchange_ids))
print("Demand reactions:", len(demand_ids))
print("Sink reactions:", len(sink_ids))
print("EX_glc_D_e_" in exchange_ids)
print("bio1" in exchange_ids)

Exchange reactions: 569
Demand reactions: 15
Sink reactions: 3
True
False


In [13]:
rid = "bio1"
if rid in exchange_ids:
    kind = "exchange"
elif rid in demand_ids:
    kind = "demand"
elif rid in sink_ids:
    kind = "sink"
else:
    kind = "other"
print(rid, "->", kind)

bio1 -> other


In [14]:
rows = []
for r in m.exchanges[:3]:
    rows.append({
        "rxn_id": r.id,
        "lb": r.lower_bound,
        "ub": r.upper_bound,
    })
print(rows)
table = pd.DataFrame(rows).set_index("rxn_id")
table

[{'rxn_id': 'EX_5mthf_e_', 'lb': -1000.0, 'ub': 1000.0}, {'rxn_id': 'EX_ala_D_e_', 'lb': -1000.0, 'ub': 1000.0}, {'rxn_id': 'EX_andrstrnglc_e_', 'lb': -1000.0, 'ub': 1000.0}]


,lb,ub
rxn_id,,
EX_5mthf_e_,-1000.0,1000.0
EX_ala_D_e_,-1000.0,1000.0
EX_andrstrnglc_e_,-1000.0,1000.0


In [15]:
table["lb"].value_counts()

lb
-1000.0    3
Name: count, dtype: int64

In [16]:
for label, model in models.items():
    census = lbp.boundary_census(model)
    print(f"\n=== {label} ===")
    print(census["Reaction type"].value_counts())
    not_exchange = census[census["Reaction type"] != "exchange"]
    print(not_exchange)


=== E_faecalis_MMH594 ===
Reaction type
exchange    173
demand       11
other         2
sink          1
Name: count, dtype: int64
                  Reaction type      Metabolite Compartment  Lower bound  \
Reaction ID                                                                
DM_5DRIB                 demand         5drib_c           c          0.0   
DM_dhptd_c_               other         dhptd_c           c      -1000.0   
dreplication             demand        dnarep_c           c          0.0   
pbiosynthesis            demand  proteinsynth_c           c          0.0   
rtranscription           demand      rnatrans_c           c          0.0   
DM_2HYMEPH               demand      phol2omt_c           c          0.0   
sink_PGPm1_c_              sink         PGPm1_c           c         -1.0   
DM_4HBA                  demand      bzalc4oh_c           c          0.0   
DM_5MTR                  demand          5mtr_c           c          0.0   
DM_HQN                   demand  

## growth_diagnosis

**Purpose**: Run a few experiments on a model. Each experiment removes one possible cause of zero growth, and every change is undone afterwards. The first experiment that restores growth names the cause.

**How is it done?**
1. As loaded -> no changes
2. Maximize. It forces the objective direction to maximum. The cause of zero growth is most likely cause by setting the objective direction to minimum.
3. All boundary open. It sets every boundary reaction bounds to (-1000, 1000). The cause of zero growth is most likely due to a closed boundary reaction.
4. Both 2 and 3

In [17]:
for label, model in models.items():
    print(label, model.objective_direction)

E_faecalis_MMH594 max
E_lenta_DSM2243 max
E_coli_Nissle1917 max
A_muciniphila_BAA835 max


In [18]:
sol = m.optimize()
print(sol.status)
print(sol.objective_value)

optimal
0.0


In [19]:
glc = m.reactions.get_by_id("EX_glc_D_e_")
print("before:", glc.lower_bound)

with m:
    glc.lower_bound = 0
    print("inside:", glc.lower_bound)

print("after:", glc.lower_bound)

before: -1000.0
inside: 0
after: -1000.0


In [20]:
print(glc.bounds)
with m:
    glc.bounds = (-5, 10)
    print(glc.bounds)
print(glc.bounds)

(-1000.0, 1000.0)
(-5, 10)
(-1000.0, 1000.0)


In [21]:
import math

for value in [0.0, float("nan"), 0.5]:
    print(value,
          math.isnan(value),
          value > 1e-6,
          not math.isnan(value) and value > 1e-6)

0.0 False False False
nan True False False
0.5 False True True


In [22]:
table = {}
for label, model in models.items():
    table[label] = lbp.growth_diagnosis(model)["Growth"]
pd.DataFrame(table)

,E_faecalis_MMH594,E_lenta_DSM2243,E_coli_Nissle1917,A_muciniphila_BAA835
Experiment,,,,
1. as loaded,0.0,0.0,0.0,0.0
2. maximize,0.0,0.0,0.0,0.0
3. all boundary open,0.0,0.0,0.0,0.0
4. both 2 and 3,0.0,0.0,0.0,0.0


## biomass_precursor_check

**Purpose**: For each reactants of the biomass reaction, ask: "If making one metabolite were the only goal, how much could the model make?". An answer of 0 means that the reactant blocks growth. This is a standard quality check for GEMs, called the biomass precursor test.

**How it works?**
For each ingredient:
1. add a temporary reaction that removes that metabolite (a drain)
2. make the drain the objective
3. maximize it

In [23]:
bio = lbp.objective_reaction(m)
print(len(bio.reactants), "ingredients")
print(len(bio.products), "products")

for met in bio.products:
    print(met.id, bio.metabolites[met])

78 ingredients
9 products
PGPm1_c 0.025011
adp_c 40.0
apoACP_c 0.0030965
biomass_c 1.0
cbi_c 0.0030965
dmbzid_c 0.0030965
h_c 40.0
pi_c 39.9969
ppi_c 0.4846


In [24]:
from cobra import Reaction                      # the "blueprint" for reactions

met = bio.reactants[0]                          # the first ingredient
drain = Reaction("TEST_DRAIN_" + met.id,        # + joins two pieces of text
                 lower_bound=0,                 # cannot run backwards (supplies nothing)
                 upper_bound=1000)              # can remove up to 1000 units
drain.add_metabolites({met: -1})                # -1 = removes one unit of met
print(drain.reaction)                           # the reaction written as an equation

10fthf_c --> 


In [25]:
with m:
    m.add_reactions([drain])
    print("inside:", drain.id in m.reactions)
print("after:", drain.id in m.reactions)

inside: True
after: False


In [26]:
met = bio.reactants[0]
drain = Reaction("TEST_DRAIN_" + met.id, lower_bound=0, upper_bound=1000)
drain.add_metabolites({met: -1})

with m:
    m.add_reactions([drain])
    m.objective = drain
    print(met.id, m.slim_optimize())
print("objective is back to:", lbp.objective_reaction(m).id)

10fthf_c 1000.0
objective is back to: bio1


In [27]:
def add_drain(model, met):
    drain = Reaction("TEST_DRAIN_" + met.id, lower_bound=0, upper_bound=1000)
    drain.add_metabolites({met: -1})
    model.add_reactions([drain])
    return drain

with m:
    d = add_drain(m, bio.reactants[1])
    m.objective = d
    print(bio.reactants[1].id, m.slim_optimize())

2dmmq8_c 1000.0


In [28]:
rows = []
for met in bio.reactants[:5]:
    with m:
        d = add_drain(m, met)
        m.objective = d
        v = m.slim_optimize()
    rows.append({"metabolite": met.id, "max_production": v})

table = pd.DataFrame(rows)
table.sort_values("max_production")

,metabolite,max_production
4,PGP_c,0.000000e+00
3,ACP_c,2.099966e-13
0,10fthf_c,1.000000e+03
1,2dmmq8_c,1.000000e+03
2,5mthf_c,1.000000e+03


In [29]:
pre = lbp.biomass_precursor_check(m)
print(pre["Producible?"].value_counts())    # how many True and how many False
pre[~pre["Producible?"]]                    # ~ means "not": keep only the unmakeable ones

Producible?
True     72
False     6
Name: count, dtype: int64


,Metabolite,Name,Coefficient,Maximum production,Producible?
3,ACP_c,Acyl carrier protein,-0.003096,2.553454e-29,False
4,PGP_c,Peptidoglycan polymer (n subunits),-0.025011,0.000000e+00,False
53,pg180_c,,-0.010648,0.000000e+00,False
52,pei17_c,,-0.010648,0.000000e+00,False
51,peai17_c,,-0.010648,0.000000e+00,False
50,pe180_c,,-0.010648,0.000000e+00,False


In [30]:
for label, model in models.items():
    pre = lbp.biomass_precursor_check(model)
    missing = pre[~pre["Producible?"]]["Metabolite"].tolist()   
    print(label, len(missing), missing)

E_faecalis_MMH594 5 ['ACP_c', 'pg180_c', 'pei17_c', 'peai17_c', 'pe180_c']
E_lenta_DSM2243 7 ['peai17_c', 'coa_c', 'pg180_c', 'pei17_c', 'pe180_c', 'cys_L_c', 'ACP_c']
E_coli_Nissle1917 6 ['ACP_c', 'PGP_c', 'pg180_c', 'pei17_c', 'peai17_c', 'pe180_c']
A_muciniphila_BAA835 8 ['2dmmq8_c', 'ACP_c', 'PGP_c', 'adocbl_c', 'pg180_c', 'pei17_c', 'peai17_c', 'pe180_c']


## rescue_test
**Purpose**: Adds a temporary reaction that supplies chosen metabolites for free, then check whether biomass can be made.

**Growth return:** the true blocker is among the metabolites you supplied.

**Still 0:** something else is missing too

In [31]:
bio = lbp.objective_reaction(m)
acp = m.metabolites.get_by_id("ACP_c")
apo = m.metabolites.get_by_id("apoACP_c")
print(bio.metabolites[acp], bio.metabolites[apo])

-0.0030965 0.0030965


In [32]:
from cobra import Reaction

met = m.metabolites.get_by_id("pe180_c")
source = Reaction("TEST_SOURCE_" + met.id, lower_bound=0, upper_bound=1000)
source.add_metabolites({met: 1})
print(source.reaction)

 --> pe180_c


In [33]:
with m:
    m.add_reactions([source])
    print("growth with pe180 supplied:", m.slim_optimize())
print("growth afterwards:", m.slim_optimize())

growth with pe180 supplied: 0.0
growth afterwards: 0.0


In [34]:
lipids = ["pg180_c", "pe180_c", "pei17_c", "peai17_c"]

with m:
    for met_id in lipids:
        met = m.metabolites.get_by_id(met_id)
        s = Reaction("TEST_SOURCE_" + met_id, lower_bound=0, upper_bound=1000)
        s.add_metabolites({met: 1})
        m.add_reactions([s])
    print("Growth with all 4 lipids supplied:", m.slim_optimize())

Growth with all 4 lipids supplied: 187.45575681789055


In [35]:
CARRIERS = ["ACP_c", "PGP_c", "adocbl_c"]                 

rows = []
for label, model in models.items():
    pre = lbp.biomass_precursor_check(model)
    missing = pre[~pre["Producible?"]]["Metabolite"].tolist()       
    suspects = [x for x in missing if x not in CARRIERS]           
    rows.append({
        "strain": label,
        "supply all missing": lbp.rescue_test(model, missing),
        "supply suspects only": lbp.rescue_test(model, suspects),
        "supply carriers only": lbp.rescue_test(model, [x for x in missing if x in CARRIERS]),
    })
pd.DataFrame(rows).set_index("strain")

,supply all missing,supply suspects only,supply carriers only
strain,,,
E_faecalis_MMH594,76.880376,76.880376,0.0
E_lenta_DSM2243,59.431166,59.431166,0.0
E_coli_Nissle1917,187.455757,187.455757,0.0
A_muciniphila_BAA835,40.777835,40.777835,0.0


## producing_reactions

**Purpose**: List the reactions that can make a given metabolite. It's the basic tool for tracing a gap backwards: "What makes X? and what makes the ingredients of that?

In [36]:
met = m.metabolites.get_by_id("glc_D_c")
print(met)

glc_D_c


In [37]:
for r in met.reactions:
    coef = r.metabolites[met]
    forward = coef > 0 and r.upper_bound > 0
    backward = coef < 0 and r.lower_bound < 0
    print(r.id, forward, backward, forward or backward)

AMALT2 True False True
AMALT3 True False True
TRE6PH True False True
MLTP4 True False True
MADGBG True False True
HEX1 False False False
BGLA True False True
AMALT4 True False True
MBDGBG True False True
GALS3 True False True
GALM1r True False True
XYLI2 False True True
S1FBFRCTSLT True False True
GLCabcpp True False True
TREH True False True
TREGLUDR2 True False True
MALT True False True
BGLA1 True False True
MALT6PH True False True
GLCt2pp True False True
G1PPTi True False True
LACZ True False True
AMALT1 True False True


In [38]:
%%writefile -a ../src/lbpscreen/analysis.py

def producing_reactions(model, met_id):
    """
    Lists the reactions that can make a metabolite, given their current bounds
    """
    met = model.metabolites.get_by_id(met_id)      # met_id --> retrieves metabolite
    rows = []
    for r in met.reactions:
        coef = r.metabolites[met]
        makes_forward = coef > 0 and r.upper_bound > 0
        makes_backward = coef < 0 and r.lower_bound < 0
        if makes_forward or makes_backward:
            rows.append({
                "Reaction ID": r.id,
                "Name": r.name,
                "Equation": r.reaction,
                "Lower bound": r.lower_bound,
                "Upper bound": r.upper_bound,
            })
    return pd.DataFrame(rows)

Appending to ../src/lbpscreen/analysis.py


In [39]:
lbp.producing_reactions(m, "glu_L_c")

,Reaction ID,Name,Equation,Lower bound,Upper bound
0,GLUN,Glutaminase,gln_L_c + h2o_c --> glu_L_c + nh4_c,0.0,1000.0
1,GLUPRT,Glutamine Phosphoribosyldiphosphate Amidotrans...,gln_L_c + h2o_c + prpp_c --> glu_L_c + ppi_c +...,0.0,1000.0
2,5GLUP5T1,(5-Glutamyl)-peptide:amino-acid 5-glutamyltran...,cyala_L_c + glu_L_c <=> glucyala_L_c + h2o_c,-1000.0,1000.0
3,CTPS2,CTP Synthase (Glutamine),atp_c + gln_L_c + h2o_c + utp_c --> adp_c + ct...,0.0,1000.0
4,GLUABUTt7pp,"4-aminobutyrate/glutamate antiport, periplasmatic",4abut_c + glu_L_p <=> 4abut_p + glu_L_c,-1000.0,1000.0
5,OHPBAT,O-Phospho-4-hydroxy-L-threonine:2-oxoglutarate...,glu_L_c + ohpb_c <=> akg_c + phthr_c,-1000.0,1000.0
6,P5CD,1-pyrroline-5-carboxylate dehydrogenase,1pyr5c_c + 2.0 h2o_c + nad_c --> glu_L_c + h_c...,0.0,1000.0
7,SDPTA,Succinyldiaminopimelate transaminase,akg_c + sl26da_c <=> glu_L_c + sl2a6o_c,-1000.0,1000.0
8,PTRCTA,Putrescine Transaminase,akg_c + ptrc_c --> 4abutn_c + glu_L_c,0.0,1000.0
9,SGDS,Succinylglutamate desuccinylase,h2o_c + succglu_c --> glu_L_c + succ_c,0.0,1000.0


In [40]:
change = {"action": "add_sink", 
          "metabolite":"pe180_c",
          "lb": -1,
          "reason": "PROVISIONAL: biomass precursor not producible (Day 3)"}
print(change["action"], change["metabolite"])

add_sink pe180_c


In [41]:
met = m.metabolites.get_by_id(change["metabolite"])
sink = Reaction("sink_" + met.id + "_curated", lower_bound=change["lb"], upper_bound=1000)
sink.add_metabolites({met: -1})
print(sink.reaction)

pe180_c <=> 


In [44]:
import cobra

CARRIERS = ["ACP_c", "PGP_c", "adocbl_c"]
REASON = "PROVISIONAL: biomass precursor not producible; prepare with proper fix"
OUT = Path("../data/processed/curated_models")
OUT.mkdir(parents=True, exist_ok=True)
Path("../curation").mkdir(exist_ok=True)

logs = []
for label, model in models.items():
    pre = lbp.biomass_precursor_check(model)
    missing = pre[~pre["Producible?"]]["Metabolite"].tolist()
    suspects = [x for x in missing if x not in CARRIERS]
    changes = [{"action": "add_sink", "metabolite": x, "lb": -1, "reason": REASON}
               for x in suspects]
    log = lbp.apply_curation(model, changes)

    if model.slim_optimize() < 1e-6:
        extra = [{"action": "add_sink", "metabolite": x, "lb": -1, "reason": REASON}
                 for x in missing if x in CARRIERS]
        log = pd.concat([log, lbp.apply_curation(model, extra)])

    log["strain"] = label
    logs.append(log)
    print(f"{label:22s} growth after curation: {model.slim_optimize():.3f}")

    for g in model.genes:                                   # keep the original gene names...
        g.name = g.notes.get("original_id", g.id)           # ...in gene.name, which SBML preserves
    cobra.io.write_sbml_model(model, str(OUT / f"{label}.xml")) 

curation_log = pd.concat(logs)
curation_log.to_csv("../curation/curation_log.csv", index=False)
curation_log
    

E_faecalis_MMH594      growth after curation: 76.880
E_lenta_DSM2243        growth after curation: 59.292
E_coli_Nissle1917      growth after curation: 93.914
A_muciniphila_BAA835   growth after curation: 40.778


,strain


In [ ]:
%%writefile -a ../tests/test_analysis.py


def test_producing_reactions():
    m = _tiny_growing_model()
    assert lbp.producing_reactions(m, "biomass_c")["rxn_id"].tolist() == ["BIO"]


def test_apply_curation():
    m = _tiny_growing_model()
    m.reactions.EX_a_e_.lower_bound = 0              # break it
    changes = [{"action": "add_sink", "metabolite": "a_e", "lb": -10, "reason": "test"}]
    log = lbp.apply_curation(m, changes)
    assert m.slim_optimize() > 0                     # the sink repairs growth
    assert len(log) == 1                             # one change, one log row
    with pytest.raises(ValueError):                  # an unknown action must raise
        lbp.apply_curation(m, [{"action": "delete_everything"}])

In [ ]:
%%writefile -a ../tests/test_models.py


CURATED = ROOT / "data/processed/curated_models"


@pytest.mark.skipif(not CURATED.exists(), reason="curated models not built yet")
def test_curated_models_grow():
    for path in CURATED.glob("*.json"):              # every saved curated model
        assert lbp.read_model(path).slim_optimize() > 1e-6, path.name